# Review agent evaluation

32 labeled experiment designs (8 clean, 24 flawed; labels correct by construction, see `eval/build_designs.py`), scored per (case, flaw) pair with 95% Wilson intervals. Method note: the committed memos are a reference run, and flaw difficulty is bounded by construction, so read the numbers as a demonstration of the harness and the memo format; regenerate live against any model with `python eval/harness.py --live`.

In [1]:
import json

import pandas as pd

results = json.load(open('eval/results/metrics.json'))
micro = results['micro']
print('generation:', results['generation']['method'],
      results['generation']['model'])
print(f"cases: {results['n_cases']}")
for k in ('precision', 'recall'):
    m = micro[k]
    print(f"micro {k}: {m['value']} "
          f"[{m['ci_low']}, {m['ci_high']}] (n={m['n']})")

generation: reference_run claude-fable-5
cases: 32
micro precision: 1.0 [0.9036, 1.0] (n=36)
micro recall: 1.0 [0.9036, 1.0] (n=36)


In [2]:
rows = {code: {'tp': b['tp'], 'fp': b['fp'], 'fn': b['fn'],
               'precision': b['precision']['value'],
               'recall': b['recall']['value']}
        for code, b in results['per_code'].items()}
pd.DataFrame(rows).T

,tp,fp,fn,precision,recall
PEEKING,9.0,0.0,0.0,1.0,1.0
MULTIPLE_COMPARISONS,6.0,0.0,0.0,1.0,1.0
CONTAMINATION,4.0,0.0,0.0,1.0,1.0
INTERFERENCE,5.0,0.0,0.0,1.0,1.0
SUBGROUP_FISHING,4.0,0.0,0.0,1.0,1.0
NOVELTY_TOO_SHORT,4.0,0.0,0.0,1.0,1.0
UNDERPOWERED,4.0,0.0,0.0,1.0,1.0


## One memo, end to end

The memo format the agent produces: recommendation first, then flags grounded in the evidence pack (numbers computed by the lab, never by the model), then the analysis plan.

In [3]:
transcripts = json.load(open('eval/results/transcripts.json'))
memo = transcripts['memos']['multi_09']
print(memo['recommendation'])
print()
for f in memo['flags']:
    print(f"[{f['severity']}] {f['code']}: {f['quantification']}")
print()
print('Analysis plan:', memo['analysis_plan'])

Reject in current form: the combination of interference, novelty too short, underpowered means the readout cannot answer the launch question; redesign before spending the traffic.

[high] INTERFERENCE: Measured in the lab: with direct effect 0.5 and spillover 0.3 (global effect 0.8), user-level randomization estimates 0.52, missing the spillover entirely; cluster randomization recovers 0.83.
[high] NOVELTY_TOO_SHORT: Measured in the lab with a decaying effect (long-run 0.10, novelty amplitude 0.40, 5-day time constant): the week-1 readout gives 0.318, a 3.2x overstatement of the long-run effect; a post-burn-in window reads 0.13.
[high] UNDERPOWERED: Evidence pack: achieved power 0.104 against a 0.80 target; 1,120 users per arm versus 19,230 required.

Analysis plan: Welch t-test on the primary metric; add CUPED with a 28-day pre-period covariate to cut variance. Keep the scheduled-look alpha-spending plan for monitoring.
